# Araz — Preprocessing: Daily Rental & Text Features

## Objective

This notebook is responsible for preprocessing the **Daily Rental Features** and **Text Features** of the Divar real-estate dataset.

The preprocessing workflow is divided into the following stages:

1. Load only the relevant columns.
2. Inspect data types and distributions.
3. Analyze missing values and determine their meaning.
4. Detect invalid or inconsistent values.
5. Clean text features.
6. Clean daily-rental numerical features.
7. Document every preprocessing decision before applying it to the full dataset.

> **Important:** Missing values will not be filled or removed blindly.  
> First, their meaning will be investigated because some missing values may indicate that a feature is not applicable to a specific type of advertisement.

## 1. Setup

First, the required libraries are imported.

- **Pandas** is used for loading, inspecting, and preprocessing tabular data.
- **NumPy** is used for numerical operations and handling missing values.
- **Pathlib** is used to manage file paths in a platform-independent way.

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path

In [ ]:
Path.cwd()

## 2. Dataset Path

The raw dataset is stored in the `data/raw` directory.

Before loading the dataset, we first verify that the file path is correct. This prevents file-loading errors and keeps the project structure organized.

In [ ]:
DATA_PATH = Path("../data/raw/Divar.csv")
DATA_PATH.exists()

## 3. Inspecting Dataset Columns

Before loading the full dataset, only the column names are read.

Since the dataset is large, reading just the header allows us to verify the available features without unnecessary memory usage.

In [ ]:
all_columns = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()

print("Number of columns:", len(all_columns))
all_columns

## 4. Selecting Araz's Features

This notebook focuses on two groups of features:

### Text Features
- `title`
- `description`

### Daily Rental Features
- `regular_person_capacity`
- `extra_person_capacity`
- `cost_per_extra_person`
- `rent_price_on_regular_days`
- `rent_price_on_special_days`
- `rent_price_at_weekends`

Before loading the data, the existence of all required columns is checked to avoid errors caused by missing or incorrectly named features.

In [ ]:
araz_columns = [
    "title",
    "description",
    "regular_person_capacity",
    "extra_person_capacity",
    "cost_per_extra_person",
    "rent_price_on_regular_days",
    "rent_price_on_special_days",
    "rent_price_at_weekends"
]

In [ ]:
missing_columns = [
    col for col in araz_columns
    if col not in all_columns
]

missing_columns

### Context Features

Some additional columns are temporarily included to provide context for the daily-rental features.

These columns are not part of the preprocessing responsibility of this notebook. They are only used to understand whether missing values in rental-related features represent actual missing data or whether those features are not applicable to certain advertisement types.

In [ ]:
candidate_helper_columns = [
    "cat2_slug",
    "cat3_slug",
    "rent_mode",
    "rent_type"
]

helper_columns = [
    col for col in candidate_helper_columns
    if col in all_columns
]

helper_columns

## 5. Loading the Required Features

The complete dataset is loaded for the selected features.

Instead of loading every column from the original CSV file, only the features required for this preprocessing task and a small number of contextual features are loaded.

This approach preserves all observations while reducing unnecessary memory consumption.

In [ ]:
selected_columns = araz_columns + helper_columns

df = pd.read_csv(
    DATA_PATH,
    usecols=selected_columns,
    low_memory=False
)

df.shape

### Loading the Full Dataset

The complete dataset is loaded using only the selected features.

`low_memory=False` is used because Pandas may otherwise infer different data types for different chunks of a large CSV file. Reading the file with this option provides more consistent dtype inference across the selected columns.

In [ ]:
print("Shape:", df.shape)

memory_gb = df.memory_usage(deep=True).sum() / (1024 ** 3)

print(f"Memory usage: {memory_gb:.2f} GB")

## 6. Initial Data Inspection

Before applying any preprocessing steps, the selected features are inspected to understand:

- their data types,
- the number of non-null values,
- the amount of missing data,
- and whether any columns have unexpected representations.

No values are modified at this stage.

In [ ]:
df.info()

In [ ]:
missing_report = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean() * 100
})

missing_report = missing_report.sort_values(
    "missing_percent",
    ascending=False
)

missing_report

In [ ]:
df.isna().mean()

## 7. Investigating Rental Context

The daily-rental features contain a very high percentage of missing values.

However, these missing values may not represent missing information. They may indicate that the corresponding advertisement is not a daily-rental listing.

Therefore, before applying any imputation or deletion strategy, the contextual rental features `rent_mode` and `rent_type` are inspected.

In [ ]:
df["rent_mode"].value_counts(dropna=False)

In [ ]:
df["rent_type"].value_counts(dropna=False)

## 8. Identifying Daily Rental Listings

The daily-rental features are populated for only a small portion of the dataset.

Before treating their missing values, we need to determine which types of advertisements actually use these features.

A row is temporarily considered related to daily-rental information if at least one of the daily-rental features contains a value.

In [ ]:
daily_columns = [
    "regular_person_capacity",
    "extra_person_capacity",
    "cost_per_extra_person",
    "rent_price_on_regular_days",
    "rent_price_on_special_days",
    "rent_price_at_weekends"
]

daily_mask = df[daily_columns].notna().any(axis=1)

In [ ]:
daily_count = daily_mask.sum()
daily_percent = daily_mask.mean() * 100

print("Rows with at least one daily-rental feature:", daily_count)
print(f"Percentage of dataset: {daily_percent:.2f}%")

In [ ]:
df.loc[daily_mask, "cat3_slug"].value_counts().head(20)

## 9. Missing Values Within Daily-Rental Listings

The previous analysis showed that daily-rental features are only applicable to a small subset of the dataset.

Therefore, the high missing percentages observed in the full dataset are mostly structural and should not automatically be treated as missing-data problems.

The next step is to analyze missing values **only within rows that contain daily-rental information**. This helps distinguish:

- values that are not applicable to non-daily listings,
- from values that are genuinely missing within daily-rental listings.

In [ ]:
daily_df = df.loc[daily_mask, daily_columns].copy()

daily_df.shape

In [ ]:
daily_missing_report = pd.DataFrame({
    "missing_count": daily_df.isna().sum(),
    "missing_percent": daily_df.isna().mean() * 100
})

daily_missing_report.sort_values(
    "missing_percent",
    ascending=False
)

## 10. Inspecting Extra Person Capacity

`extra_person_capacity` is expected to represent a numerical capacity, but Pandas inferred it as a string column.

Before converting its data type, the unique values are inspected to determine whether the column contains:

- numerical values stored as text,
- special categorical values,
- inconsistent representations,
- or invalid entries.

No conversion is applied before understanding the existing values.

In [ ]:
df.loc[daily_mask, "extra_person_capacity"].value_counts(
    dropna=False
).head(30)

In [ ]:
df.loc[daily_mask, "extra_person_capacity"].unique()[:50]

### Investigating Non-Numeric Capacity Values

Although `extra_person_capacity` conceptually represents a numerical quantity, it is stored as a string column.

Most values appear to be numeric strings, but special values such as `30+` are also present.

Before converting the column to a numeric data type, we identify all values that cannot be parsed as numbers. This prevents losing information during conversion.

In [ ]:
extra_raw = (
    df.loc[daily_mask, "extra_person_capacity"]
    .astype("string")
    .str.strip()
)

extra_numeric_test = pd.to_numeric(
    extra_raw,
    errors="coerce"
)

non_numeric_mask = (
    extra_raw.notna()
    & extra_numeric_test.isna()
)

extra_raw[non_numeric_mask].value_counts()

### Capacity Distribution Check

Only one non-numeric value was found in `extra_person_capacity`: `30+`.

Before deciding how to encode this value, the numerical distribution of the column is inspected.

This helps determine whether `30+` represents a capped upper category and whether replacing it with a numerical value such as `30` would be reasonable.

In [ ]:
extra_capacity_temp = pd.to_numeric(
    extra_raw.replace("30+", "30"),
    errors="coerce"
)

extra_capacity_temp.describe()

In [ ]:
extra_capacity_temp.value_counts().sort_index().tail(15)

### Checking the Upper Capacity Boundary

Before converting `30+` to a numerical value, we verify whether the exact value `30` also exists in the original data.

This is important because `30` and `30+` do not have exactly the same meaning.  
Merging them without checking could result in loss of information.


In [ ]:
extra_raw[
    extra_raw.isin(["28", "29", "30", "30+"])
].value_counts()

In [ ]:
df.loc[
    df["extra_person_capacity"].eq("30+"),
    [
        "cat3_slug",
        "regular_person_capacity",
        "extra_person_capacity",
        "cost_per_extra_person"
    ]
].head(10)

### Cleaning `extra_person_capacity`

The inspection showed that `30+` is the only non-numeric value in this feature and the exact value `30` is not stored as a separate raw category.

Therefore, `30+` is encoded as `30` as a conservative capped representation. All other numeric strings are converted to numbers and the feature is stored as Pandas nullable `Int64`, preserving missing values.


In [ ]:
df["extra_person_capacity"] = (
    df["extra_person_capacity"]
    .astype("string")
    .str.strip()
    .replace("30+", "30")
    .pipe(pd.to_numeric, errors="coerce")
    .astype("Int64")
)

print("dtype:", df["extra_person_capacity"].dtype)
print("missing:", df["extra_person_capacity"].isna().sum())


## 12. Profiling Daily Rental Numerical Features

The remaining daily-rental numerical features are inspected together to identify:

- invalid negative values,
- zero values,
- fractional values,
- extreme values,
- and overall distribution patterns.

This allows the preprocessing strategy to be defined consistently across all daily-rental features.

In [ ]:
numeric_daily_cols = [
    "regular_person_capacity",
    "cost_per_extra_person",
    "rent_price_on_regular_days",
    "rent_price_on_special_days",
    "rent_price_at_weekends"
]

profile = []

for col in numeric_daily_cols:
    s = df.loc[daily_mask, col]

    profile.append({
        "column": col,
        "count": s.notna().sum(),
        "missing": s.isna().sum(),
        "min": s.min(),
        "median": s.median(),
        "mean": s.mean(),
        "max": s.max(),
        "negative_count": (s < 0).sum(),
        "zero_count": (s == 0).sum(),
        "fractional_count": (
            s.notna() & (s % 1 != 0)
        ).sum()
    })

profile_df = pd.DataFrame(profile)

profile_df

In [ ]:
df.loc[
    daily_mask,
    numeric_daily_cols
].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).T

In [ ]:
df.loc[
    daily_mask,
    "regular_person_capacity"
].value_counts().sort_index().tail(20)

### Cleaning `regular_person_capacity`

The feature contains only positive integer values between 1 and 50.

No negative, zero, or fractional values were found. Values near the upper boundary also occur repeatedly, so they are not treated as errors based only on their magnitude.

The feature is therefore converted to a nullable integer type without removing any observations.

In [ ]:
df["regular_person_capacity"] = (
    df["regular_person_capacity"]
    .astype("Int64")
)

## 13. Investigating Suspicious Daily Rental Prices

The daily-rental price features are highly right-skewed and contain extremely large values compared with their typical ranges.

For example, the maximum values are several orders of magnitude larger than the 99th percentile.

Very small values, including zero, are also present.

Before removing or replacing any values, both tails of the distributions are inspected to distinguish plausible prices from likely data-entry errors or placeholder values.

In [ ]:
price_cols = [
    "cost_per_extra_person",
    "rent_price_on_regular_days",
    "rent_price_on_special_days",
    "rent_price_at_weekends"
]

for col in price_cols:
    s = df.loc[daily_mask, col].dropna()

    print(f"\n{'='*60}")
    print(col)

    print("\nSmallest values:")
    print(s.value_counts().sort_index().head(15))

    print("\nLargest values:")
    print(s.sort_values(ascending=False).head(15).to_list())

In [ ]:
suspicious_report = []

for col in price_cols:
    s = df.loc[daily_mask, col]

    suspicious_report.append({
        "column": col,
        "zero": (s == 0).sum(),
        "<= 1,000": (s <= 1_000).sum(),
        "<= 10,000": (s <= 10_000).sum(),
        "> 10,000,000": (s > 10_000_000).sum(),
        "> 100,000,000": (s > 100_000_000).sum(),
        "> 1,000,000,000": (s > 1_000_000_000).sum(),
    })

pd.DataFrame(suspicious_report)

## 14. Cleaning Daily Rental Price Features

The daily-rental price features contain suspicious values at both ends of their distributions.

Extremely large values are several orders of magnitude larger than the 99th percentile and include clearly irregular patterns such as repeated digits. These values are treated as invalid observations.

Very small values are also suspicious; however, their original meaning is uncertain. They may represent data-entry conventions or unit inconsistencies. Therefore, they are flagged but not automatically modified.

Missing values are preserved because they may represent optional or non-applicable information rather than true missing data.

In [ ]:
price_cols = [
    "cost_per_extra_person",
    "rent_price_on_regular_days",
    "rent_price_on_special_days",
    "rent_price_at_weekends"
]

EXTREME_PRICE_THRESHOLD = 1_000_000_000

for col in price_cols:
    invalid_high = df[col] > EXTREME_PRICE_THRESHOLD
    df.loc[invalid_high, col] = np.nan

In [ ]:
for col in price_cols:
    df[f"{col}_suspicious_low"] = (
        df[col].notna() &
        (df[col] <= 10_000)
    )

In [ ]:
df["regular_person_capacity"] = (
    df["regular_person_capacity"]
    .astype("Int64")
)

In [ ]:
df[
    [
        "regular_person_capacity",
        "extra_person_capacity",
        *price_cols
    ]
].describe().T

In [ ]:
df[price_cols].max()

# Text Features Preprocessing

The dataset contains two main text features:

- `title`
- `description`

Text preprocessing will focus on basic normalization and data-quality checks while preserving useful semantic information.

Aggressive operations such as removing numbers, punctuation, or words are avoided at this stage because they may remove meaningful information.

Special attention is also given to potential target leakage, since real-estate advertisements may explicitly mention prices inside their text.

In [ ]:
text_cols = ["title", "description"]

text_report = pd.DataFrame({
    "missing": df[text_cols].isna().sum(),
    "unique": df[text_cols].nunique(),
    "empty_string": [
        df[col].fillna("").str.strip().eq("").sum()
        for col in text_cols
    ]
})

text_report

In [ ]:
for col in text_cols:
    lengths = df[col].fillna("").str.len()

    print(f"\n{col}")
    print(lengths.describe(
        percentiles=[0.01, 0.5, 0.95, 0.99]
    ))

## 15. Cleaning Text Features

The `title` and `description` columns are almost complete and contain highly diverse textual information.

The preprocessing strategy is intentionally conservative. The goal is to normalize formatting inconsistencies without removing potentially useful semantic information.

The following operations are applied:

- leading and trailing whitespace is removed,
- repeated whitespace is normalized,
- Persian/Arabic character variants are standardized,
- empty strings are converted to missing values.

Numbers and punctuation are preserved because they may contain useful information.

However, text features must later be checked for potential **target leakage**, since advertisements may explicitly mention prices in their title or description.

In [ ]:
text_cols = ["title", "description"]

for col in text_cols:
    df[col] = (
        df[col]
        .astype("string")
        .str.replace("ي", "ی", regex=False)
        .str.replace("ك", "ک", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace("", pd.NA)
    )

In [ ]:
text_validation = pd.DataFrame({
    "missing": df[text_cols].isna().sum(),
    "unique": df[text_cols].nunique(),
    "empty_string": [
        df[col].fillna("").str.strip().eq("").sum()
        for col in text_cols
    ]
})

text_validation

In [ ]:
for col in text_cols:
    print(
        col,
        "duplicated values:",
        df[col].duplicated().sum()
    )

In [ ]:
df["title_length"] = df["title"].str.len()
df["description_length"] = df["description"].str.len()

df["title_word_count"] = df["title"].str.split().str.len()
df["description_word_count"] = df["description"].str.split().str.len()

### Text Length Features

Additional lightweight features are created from the text columns:

- title length,
- description length,
- title word count,
- description word count.

These features preserve simple information about advertisement verbosity without requiring a full NLP pipeline.

### Potential Target Leakage in Text

Because the final machine-learning task includes price prediction, textual features may contain explicit price information.

Using such information directly could introduce target leakage. Therefore, potential price-related patterns are inspected before deciding whether raw text should be used in the regression model.

In [ ]:
price_keywords = r"قیمت|تومان|میلیون|میلیارد"

for col in text_cols:
    count = (
        df[col]
        .fillna("")
        .str.contains(price_keywords, regex=True)
        .sum()
    )

    print(f"{col}: {count}")


# 16. Final Validation and Integration Notes

Before handing this preprocessing work to the team, the final schema and key quality checks are reviewed.

Important decisions:

- Structural missing values outside daily-rental listings are preserved.
- Missing optional daily-rental values are not blindly imputed.
- `regular_person_capacity` and `extra_person_capacity` are represented as nullable integers.
- Clearly extreme daily-rental price values are replaced with missing values using the documented conservative rule.
- Suspiciously small price values are flagged rather than guessed or rescaled.
- Persian/Arabic character variants and whitespace are normalized in text.
- Lightweight text-length features are created.
- Raw text should **not automatically be used in the final price-regression model** before the team evaluates target-leakage risk.


In [ ]:
final_columns = [
    "title",
    "description",
    "title_length",
    "description_length",
    "title_word_count",
    "description_word_count",
    "regular_person_capacity",
    "extra_person_capacity",
    "cost_per_extra_person",
    "rent_price_on_regular_days",
    "rent_price_on_special_days",
    "rent_price_at_weekends",
]

print("Final selected dtypes:")
display(df[final_columns].dtypes.to_frame("dtype"))

print("\nFinal missing percentages:")
display(
    (df[final_columns].isna().mean() * 100)
    .sort_values(ascending=False)
    .to_frame("missing_percent")
)

assert str(df["regular_person_capacity"].dtype) == "Int64"
assert str(df["extra_person_capacity"].dtype) == "Int64"
assert df["title"].fillna("").str.strip().eq("").sum() == df["title"].isna().sum()
assert df["description"].fillna("").str.strip().eq("").sum() == df["description"].isna().sum()

print("\nFinal validation passed.")


## Team Integration

The notebook above documents the analysis and reasoning. For merging into the team's main preprocessing pipeline, the same transformations are provided below as a reusable function.

The raw dataset itself should **not** be committed to GitHub.


In [ ]:
def preprocess_araz_features(data):
    out = data.copy()

    # Capacity features
    out["regular_person_capacity"] = pd.to_numeric(
        out["regular_person_capacity"], errors="coerce"
    ).astype("Int64")

    out["extra_person_capacity"] = (
        out["extra_person_capacity"]
        .astype("string")
        .str.strip()
        .replace("30+", "30")
        .pipe(pd.to_numeric, errors="coerce")
        .astype("Int64")
    )

    # Daily-rental prices
    price_cols = [
        "cost_per_extra_person",
        "rent_price_on_regular_days",
        "rent_price_on_special_days",
        "rent_price_at_weekends",
    ]

    extreme_price_threshold = 1_000_000_000

    for col in price_cols:
        out.loc[out[col] > extreme_price_threshold, col] = np.nan
        out[f"{col}_suspicious_low"] = out[col].notna() & (out[col] <= 10_000)

    # Text normalization
    text_cols = ["title", "description"]

    for col in text_cols:
        out[col] = (
            out[col]
            .astype("string")
            .str.replace("ي", "ی", regex=False)
            .str.replace("ك", "ک", regex=False)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
            .replace("", pd.NA)
        )

    # Lightweight text features
    out["title_length"] = out["title"].str.len()
    out["description_length"] = out["description"].str.len()
    out["title_word_count"] = out["title"].str.split().str.len()
    out["description_word_count"] = out["description"].str.split().str.len()

    return out
